# SpendWise AI · laboratorio actualizado

La entrega principal es la web app (`python app.py`) y la presentación `/pitch`.
Este notebook comparte el código de la aplicación. El notebook anterior y sus outputs se preservan en `docs/SpendWiseAI_historico.ipynb`.

**Frontera:** IA extrae → código valida → persona revisa → código calcula.
**Evidencia:** histórico Gemini 8/11; software offline 11/11 con fixtures manuales. No son métricas equivalentes.

Ejecutar desde la raíz del proyecto. En Colab hay que subir también `spendwise_core.py`, `spendwise_service.py` y la carpeta `evals/`, no solo el notebook.


In [ ]:
from pathlib import Path
import json
from spendwise_core import validate_financial_output
from spendwise_service import prepare_review, confirm_review, call_gemini
from evals.run_evals import load_cases, load_fixtures, run_suite

cases = load_cases()
fixtures = load_fixtures()
print('Núcleo cargado. No se ha llamado a Gemini.')


## 1. Usuario, hipótesis y baseline

Estudiantes y jóvenes profesionales con gastos en texto informal. La hipótesis es reducir el esfuerzo de organización frente a una hoja de cálculo, sin aumentar errores finales. No se han realizado pruebas con usuarios; el protocolo está en `docs/validacion_usuarios.md`.

La elección de presentación es una web app local de tres pasos. La arquitectura y los gates propuestos están en `docs/arquitectura.md` y `docs/gates.md`.


## 2. Ensayo explícitamente simulado

La siguiente extracción fue anotada manualmente, no generada en esta ejecución por Gemini.


In [ ]:
case = cases[0]
review = prepare_review(fixtures[case['id']], case['input'], {'mode': 'fixture'})
print(case['input'])
print(json.dumps(review, ensure_ascii=False, indent=2))


## 3. Revisión humana

Revisar antes de ejecutar. Para la demostración con botones y tabla editable, usar la web app.
La variable `confirmed` queda en False deliberadamente: cambiarla solo después de revisar ingreso, movimientos e incidencias.


In [ ]:
import copy
edited = copy.deepcopy(review['movimientos'])
confirmed = False
if confirmed:
    result = confirm_review(review, {
        'confirmed': confirmed,
        'ingreso_total': review['ingreso_total'],
        'movimientos': edited,
        'selected_ids': [],
    })
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print('Pendiente: revisión y confirmación humana. No se generó un resultado final.')


## 4. Evals offline reproducibles

No llaman a Gemini. Evalúan controles de software frente a extracciones manuales, incluyendo casos adversariales. Se guardan todos los resultados, no solamente el score.


In [ ]:
offline = run_suite(live=False)
print(f"Software con fixtures: {offline['passed']}/{offline['total']}")
for result in offline['results']:
    print(result['case'], result['status'], result['errors'])


## 5. Gemini real (opcional, requiere clave y cuota)

No pegar claves en el notebook ni en outputs. Usar entrada oculta. Activar `RUN_LIVE` solo si se quiere enviar el texto a Gemini. Un error no se reemplaza por simulación.


In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    import os
    from getpass import getpass
    if not os.getenv('GEMINI_API_KEY'):
        os.environ['GEMINI_API_KEY'] = getpass('Gemini API key (oculta): ')
    extraction, metadata = call_gemini(case['input'])
    live_review = prepare_review(extraction, case['input'], metadata)
    print(json.dumps(live_review, ensure_ascii=False, indent=2))
else:
    print('Gemini real no ejecutado.')


## 6. Evaluación real repetida

Once casos × tres repeticiones = 33 llamadas. Guardar evidencia y comparar con el mismo dataset; no presentar una ejecución offline como resultado de Gemini.


In [ ]:
RUN_LIVE_EVALS = False
if RUN_LIVE_EVALS:
    import os
    from getpass import getpass
    if not os.getenv('GEMINI_API_KEY'):
        os.environ['GEMINI_API_KEY'] = getpass('Gemini API key (oculta): ')
    live = run_suite(live=True, repeat=3)
    Path('evals/runs').mkdir(exist_ok=True)
    Path('evals/runs/live.json').write_text(json.dumps(live, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"Gemini real: {live['passed']}/{live['total']}")
else:
    print('Evaluación real pendiente; no hay un score nuevo de Gemini.')


## 7. Flujo y presentación

```mermaid
flowchart LR
    T[Texto] --> G[Gemini: extracción]
    G --> V[Validar esquema e incidencias]
    V --> H[Revisión y confirmación humana]
    H --> C[Cálculo determinista]
    C --> S[Escenario elegido por usuario]
    S --> O[Validación final y presupuesto]
```

Pitch de seis minutos: `docs/pitch_6_minutos.md`, presentación navegable `/pitch` y versión PDF en `docs/SpendWiseAI_pitch.pdf` cuando se exporta.

Antes de exponer: ejecutar `python verify.py`, revisar gates pendientes y ensayar el caso normal más un dato faltante. Ninguna prueba automática reemplaza entrevistas o la rúbrica del profesor.
